In [ ]:
import lightning as L
import torch
from IPython.display import Video

from training.evaluate import evaluate, record_video
from training.sim import SKILLS, DuckEnv, hardware
from training.ppo import ActorCritic, PolicyModule, RolloutData, compute_gae, flat, normalize

In [ ]:
hw = hardware.load("2026-11-07-hd1910")
env = DuckEnv(SKILLS["walk"], hw, num_envs=2048, device="cuda", seed=1)
env.reset()
env.randomize_episode_starts()                           # 回合进度打散，和正式训练一样
net = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim).to("cuda")   # 61 个观测进，14 个关节的动作出

In [ ]:
class PPOData(RolloutData):                              # data 一半
    gamma, lam = 0.99, 0.95
    num_learning_epochs, num_mini_batches = 5, 4

    def prepare(self, rollout):
        rewards = rollout["rewards"] + self.gamma * rollout["values"].squeeze(-1) * rollout["time_outs"]
        advantages, returns = compute_gae(rewards, rollout["dones"], rollout["values"], rollout["next_value"],
                                          self.gamma, self.lam)
        return {**flat(rollout, "obs", "critic_obs", "actions", "log_probs", "values", "action_means", "action_stds"),
                "advantages": normalize(advantages).flatten(), "returns": returns.flatten(0, 1)}

    def batches(self, samples):
        size = len(samples["actions"]) // self.num_mini_batches
        order = torch.randperm(self.num_mini_batches * size, device=samples["obs"].device)   # 只打乱一次
        picks = [order[i * size:(i + 1) * size] for i in range(self.num_mini_batches)]
        return [{key: value[p] for key, value in samples.items()}
                for _ in range(self.num_learning_epochs) for p in picks]

In [ ]:
class PPO(PolicyModule):                                 # model 一半
    learning_rate, clip_param, desired_kl = 1.0e-3, 0.2, 0.01
    value_loss_coef, entropy_coef = 1.0, 0.01

    def __init__(self, net):
        super().__init__(net)
        self.kl, self.lr = torch.zeros(()), self.learning_rate

    def loss(self, batch):
        dist = self.net.action_distribution(batch["obs"])
        log_probs = dist.log_prob(batch["actions"]).sum(-1)
        entropy = dist.entropy().sum(-1).mean()
        values = self.net.value(batch["critic_obs"])
        with torch.no_grad():
            old = torch.distributions.Normal(batch["action_means"], batch["action_stds"])
            self.kl = torch.distributions.kl_divergence(old, dist).sum(-1).mean()
        ratio = torch.exp(log_probs - batch["log_probs"])
        clipped = torch.clamp(ratio, 1 - self.clip_param, 1 + self.clip_param)
        surrogate = torch.max(-batch["advantages"] * ratio, -batch["advantages"] * clipped).mean()
        value_clipped = batch["values"] + (values - batch["values"]).clamp(-self.clip_param, self.clip_param)
        value_loss = torch.max((values - batch["returns"]).pow(2), (value_clipped - batch["returns"]).pow(2)).mean()
        return surrogate + self.value_loss_coef * value_loss - self.entropy_coef * entropy

    def on_before_optimizer_step(self, optimizer):
        kl = float(self.kl)
        if kl > self.desired_kl * 2.0:
            self.lr = max(1.0e-5, self.lr / 1.5)
        elif 0.0 < kl < self.desired_kl / 2.0:
            self.lr = min(1.0e-2, self.lr * 1.5)
        for group in optimizer.param_groups:
            group["lr"] = self.lr

In [ ]:
import training.ppo as official

rollout = official.sample_rollout(env, net, 24)          # 采一轮
torch.manual_seed(0)
mine = PPOData(env, net).batches(PPOData(env, net).prepare(rollout))
torch.manual_seed(0)
ref = official.PPOData(env, net).batches(official.PPOData(env, net).prepare(rollout))
all(torch.equal(PPO(net).loss(a), official.PPO(net).loss(b)) for a, b in zip(mine, ref))

In [ ]:
before = evaluate(net, SKILLS["walk"], hw)               # 512 只鸭子各走 20 秒
before["falls"], round(before["mean_step_reward"], 3)

In [ ]:
# 每轮重建一次数据，也就是每轮用最新的网络重新采样
trainer = L.Trainer(max_epochs=1500, reload_dataloaders_every_n_epochs=1,
                    accelerator="gpu", devices=1, logger=False, enable_checkpointing=False)
trainer.fit(PPO(net), PPOData(env, net))

In [ ]:
after = evaluate(net, SKILLS["walk"], hw)
after["falls"], round(after["mean_step_reward"], 3)

In [ ]:
def load(path):
    old = ActorCritic(env.obs_dim, env.critic_obs_dim, env.action_dim)
    old.load_state_dict(torch.load(path))
    return evaluate(old, SKILLS["walk"], hw)


for name, path in (("v1 REINFORCE", "../S03-policy-gradient/v1_reinforce.pt"), ("v2 A2C", "../S04-actor-critic/v2_a2c.pt")):
    result = load(path)
    print(name, result["falls"], round(result["mean_step_reward"], 3))
print("v3 PPO", after["falls"], round(after["mean_step_reward"], 3))

In [ ]:
record_video(net, SKILLS["walk"], hw, "v3_ppo.mp4")
Video("v3_ppo.mp4", embed=True)

In [ ]:
torch.save(net.state_dict(), "v3_ppo.pt")